# 02. Isotope Calculator

**목표**: `echo_tof.isotope_calc` 모듈의 `IsotopicDistributionCalculator`를 이해한다.

자연계의 원소는 여러 동위원소(isotope)를 갖는다.  
예를 들어 탄소는 12C(98.93%)와 13C(1.07%)가 존재하므로,
분자의 질량 스펙트럼에는 M, M+1, M+2, ... 피크가 나타난다.

이 계산기는  **convolution + binary exponentiation** 알고리즘을 사용한다.

## 환경 설정

In [ ]:
import os, sys
os.chdir(r'C:\Users\gogoc\AppData\Local\Temp\echo-tof-verify')
sys.path.insert(0, '.')

## 1. Convolution의 기본 원리

두 원자의 동위원소 분포를 합치려면 모든 조합의:
- **질량** = 합
- **확률** = 곱

을 계산한 뒤, 비슷한 질량의 피크를 병합(merge)한다.

```python
# IsotopicPeak.convolute() — 핵심 연산
@staticmethod
def convolute(item1, item2) -> IsotopicPeak:
    return IsotopicPeak(
        mass=item1.mass + item2.mass,         # 질량: 덧셈
        abundance=item1.abundance * item2.abundance,  # 확률: 곱셈
    )
```

C6 분자의 패턴을 직접 계산하려면 C 1개의 분포를 6번 self-convolute해야 한다.
**Binary exponentiation**이 이를 `O(log N)`으로 줄인다.

## 2. Binary Exponentiation 최적화

원소 N개의 분포를 구할 때, N을 이진 분해한다:

예: C7 = C4 * C2 * C1 (7 = 4 + 2 + 1)

```python
def _calculate_element_array(self, ec: ElementCount) -> IsotopicArray:
    result = None
    count = ec.count  # 예: 7
    power = 1
    while count != 0:
        if count % 2 != 0:  # 이진수의 현재 비트가 1이면
            power_array = self._get_element_array_power_of_two(ec.element, power)
            if result is None:
                result = power_array
            else:
                result = result.convolute(power_array)
        count //= 2
        power *= 2
    return result
```

2의 거듭제곱 분포는 **재귀적 제곱**으로 구한다:
- C1: 원소의 기본 동위원소 분포
- C2 = C1 * C1
- C4 = C2 * C2

결과는 캐시에 저장하여 재사용한다.

## 3. IsotopicArray와 피크 병합

`IsotopicArray`는 피크를 질량 순서로 유지하면서, tolerance 범위 내 피크를 자동 병합한다.

```python
def merge(self, other: IsotopicPeak):
    total = self.abundance + other.abundance
    if total > 0:
        self.mass = (self.mass * self.abundance +
                     other.mass * other.abundance) / total  # 가중평균
        self.abundance = total
```

병합 기준은 Da 또는 ppm tolerance로 설정할 수 있다.

## 4. 단일 원소의 동위원소 확인

계산기가 사용하는 원소별 동위원소 데이터를 확인해 보자.

In [ ]:
from echo_tof.elements import PeriodicTable

pt = PeriodicTable.instance()
for sym in ['C', 'H', 'N', 'O']:
    elem = pt.get_element(sym)
    print(f"\n{elem.symbol} ({elem.name}):")
    for iso in elem.isotopes:
        if iso.abundance > 0:
            print(f"  mass={iso.mass:.6f}  abundance={iso.abundance:.6f} ({iso.abundance*100:.3f}%)")

## 5. 예제: TNT (C7H5N3O6)의 동위원소 패턴

TNT의 이론적 isotope pattern을 계산하고, M, M+1, M+2, M+3 피크를 확인한다.

In [ ]:
from echo_tof.isotope_calc import IsotopicDistributionCalculator

idc = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)
peaks = idc.calculate('C7H5N3O6')

print(f"{'Peak':>6s}  {'Mass (Da)':>12s}  {'Abundance':>10s}  {'Normalised':>10s}")
print("-" * 44)
for i, p in enumerate(peaks):
    label = f"M+{i}" if i > 0 else "M"
    print(f"{label:>6s}  {p.mass:12.6f}  {p.abundance:10.6f}  {p.normalised_abundance*100:9.2f}%")

## 6. 패턴 시각화 (텍스트 기반)

matplotlib 없이도 패턴의 상대 강도를 확인할 수 있다.

In [ ]:
print("TNT (C7H5N3O6) Isotope Pattern")
print("=" * 50)
for i, p in enumerate(peaks):
    label = f"M+{i}" if i > 0 else "M  "
    bar_len = int(p.normalised_abundance * 40)
    print(f"{label}  {p.mass:10.4f} Da  {'#' * bar_len} {p.normalised_abundance*100:.1f}%")

## 7. 큰 분자 비교: Caffeine vs Insulin chain

분자가 커질수록 M+1, M+2 피크의 상대 강도가 증가한다.

In [ ]:
molecules = {
    'Caffeine (C8H10N4O2)': 'C8H10N4O2',
    'Reserpine (C33H40N2O9)': 'C33H40N2O9',
}

for name, formula in molecules.items():
    peaks = idc.calculate(formula)
    print(f"\n{name}:")
    for i, p in enumerate(peaks[:5]):
        label = f"M+{i}" if i > 0 else "M  "
        print(f"  {label}  {p.normalised_abundance*100:6.2f}%")

## 8. 캐시 효과 확인

동일한 원소+개수 조합은 캐시에서 즉시 반환된다.

In [ ]:
import time

idc2 = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)

# 첫 번째 호출 (캐시 비어있음)
t0 = time.perf_counter()
_ = idc2.calculate('C33H40N2O9')
t1 = time.perf_counter()
print(f"첫 번째 호출: {(t1-t0)*1000:.3f} ms")

# 두 번째 호출 (캐시 히트)
t0 = time.perf_counter()
_ = idc2.calculate('C33H40N2O9')
t1 = time.perf_counter()
print(f"두 번째 호출: {(t1-t0)*1000:.3f} ms")

print(f"\n캐시 항목 수: {len(idc2._cache)}")

## 정리

| 구성요소 | 역할 |
|----------|------|
| `IsotopicPeak` | 단일 피크 (mass + abundance) |
| `IsotopicArray` | 피크 배열, tolerance 내 자동 병합 |
| `IsotopicDistributionCalculator` | 엔진: binary exponentiation + 캐시 |

**핵심 알고리즘**: 원소별 분포를 binary exponentiation으로 계산한 뒤,
원소 간 convolution으로 전체 분자의 isotope pattern을 구한다.

다음 노트북에서는 이 이론 패턴을 실측 데이터와 **비교(pattern matching)** 하는 방법을 다룬다.